# 23a — Case Studies: Image Classification and Object Detection

Corresponds to: Note 23 — Case Studies: Image Classification and Object Detection

**Libraries introduced in this notebook:** No new libraries — PyTorch and torchvision continue from Note 22.

**What this notebook demonstrates:**
1. Full image classification pipeline on CIFAR-10 — end to end from raw image to predicted label
2. IoU calculation — the core metric for evaluating bounding box quality
3. YOLO output tensor structure — building and interpreting the S×S×(B×5+C) tensor
4. Non-maximum suppression — removing duplicate detections by hand

## Part 1 — Full Classification Pipeline on CIFAR-10

Note 23 Part 1 describes the classification pipeline: conv blocks → flatten → FC → softmax → argmax.
This part implements that pipeline end to end and inspects each stage.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Load CIFAR-10. Standard normalisation with ImageNet-style mean and std.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)

classes = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print('Train samples:', len(train_set))
print('Test samples: ', len(test_set))

Define the CNN. Two conv blocks followed by FC layers — the standard pipeline from Note 23 Part 1.

In [ ]:
class ClassificationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Feature extractor
        self.conv1 = nn.Conv2d(3,  32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        # Classifier
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.drop  = nn.Dropout(0.4)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = x.view(x.size(0), -1)                       # flatten
        x = self.drop(F.relu(self.fc1(x)))
        x = self.fc2(x)                                  # raw logits
        return x

model = ClassificationCNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print('Total parameters:', total_params)

Train for 10 epochs.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f'Epoch {epoch+1:2d}  loss: {running_loss/len(train_loader):.4f}')

Evaluate on the test set and inspect the softmax output for a batch of images.

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()

print(f'Test accuracy: {100 * correct / len(test_set):.2f}%')

Inspect the full pipeline output for 6 test images: raw logits → softmax probabilities → argmax prediction.

In [ ]:
images, labels = next(iter(test_loader))
images_dev = images[:6].to(device)

model.eval()
with torch.no_grad():
    logits = model(images_dev)
    probs  = F.softmax(logits, dim=1)
    preds  = probs.argmax(dim=1)

mean = torch.tensor([0.4914, 0.4822, 0.4465])
std  = torch.tensor([0.2470, 0.2435, 0.2616])

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    img = images[i] * std[:, None, None] + mean[:, None, None]
    img = img.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    color = 'green' if preds[i] == labels[i] else 'red'
    axes[0, i].set_title(f'pred: {classes[preds[i]]}\ntrue: {classes[labels[i]]}',
                         fontsize=8, color=color)
    axes[1, i].bar(range(10), probs[i].cpu().numpy(), color='steelblue')
    axes[1, i].set_xticks(range(10))
    axes[1, i].set_xticklabels(classes, rotation=90, fontsize=6)
    axes[1, i].set_ylim(0, 1)
    if i == 0:
        axes[1, i].set_ylabel('P(class)', fontsize=8)

plt.suptitle('Classification pipeline: image -> softmax probabilities -> argmax', fontsize=11)
plt.tight_layout()
plt.show()
print('Top-1 predictions shown above')

## Part 2 — IoU Calculation

IoU (Intersection over Union) is the standard metric for bounding box quality.
This part implements IoU and verifies the worked example from Exercise 23 Section C.

```
Ground truth box A: (x_min=0, y_min=0, x_max=6, y_max=4)  area = 24
Predicted box B:    (x_min=2, y_min=1, x_max=8, y_max=5)  area = 24
```

In [ ]:
def iou(boxA, boxB):
    """Compute IoU for two boxes in (x_min, y_min, x_max, y_max) format."""
    ix1 = max(boxA[0], boxB[0])
    iy1 = max(boxA[1], boxB[1])
    ix2 = min(boxA[2], boxB[2])
    iy2 = min(boxA[3], boxB[3])

    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union, inter, union, areaA, areaB

A = (0, 0, 6, 4)
B = (2, 1, 8, 5)

score, inter, union, aA, aB = iou(A, B)

print(f'Box A area:          {aA}')
print(f'Box B area:          {aB}')
print(f'Intersection area:   {inter}')
print(f'Union area:          {union}')
print(f'IoU:                 {inter}/{union} = {score:.4f}')
print(f'Correct detection?   {score >= 0.5}  (threshold = 0.5)')

Visualise the two boxes and their intersection.

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(-0.5, 6.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('x'); ax.set_ylabel('y')

rA = mpatches.Rectangle((A[0], A[1]), A[2]-A[0], A[3]-A[1],
    linewidth=2.5, edgecolor='#4C72B0', facecolor='#4C72B0', alpha=0.2, label='Ground truth A')
rB = mpatches.Rectangle((B[0], B[1]), B[2]-B[0], B[3]-B[1],
    linewidth=2.5, edgecolor='#C44E52', facecolor='#C44E52', alpha=0.2, label='Predicted B')

ix1 = max(A[0], B[0]); iy1 = max(A[1], B[1])
ix2 = min(A[2], B[2]); iy2 = min(A[3], B[3])
rI = mpatches.Rectangle((ix1, iy1), ix2-ix1, iy2-iy1,
    linewidth=0, facecolor='#55A868', alpha=0.5, label=f'Intersection = {inter}')

ax.add_patch(rA)
ax.add_patch(rB)
ax.add_patch(rI)
ax.legend(fontsize=9)
ax.set_title(f'IoU = {inter}/{union} = {score:.4f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'IoU = {score:.4f} — below 0.5 threshold, prediction not counted as correct')

## Part 3 — YOLO Output Tensor Structure

YOLO produces a fixed output tensor of shape S × S × (B×5 + C).
This part builds a dummy YOLO output tensor and reads back the predictions for one cell.

In [ ]:
S = 7    # grid size
B = 2    # boxes per cell
C = 10   # classes (CIFAR-10)

output_depth = B * 5 + C
print(f'YOLO output tensor shape: {S} x {S} x {output_depth}')
print(f'  B*5 = {B}*5 = {B*5}  (cx, cy, w, h, conf per box)')
print(f'  C   = {C}  (class scores)')
print(f'  Total depth: {output_depth}')

# Simulate a raw output tensor (random, as if from an untrained network)
torch.manual_seed(0)
raw_output = torch.randn(S, S, output_depth)
print(f'\nRaw output tensor shape: {raw_output.shape}')

Parse the predictions for grid cell (3, 4) — row 3, column 4.

In [ ]:
row, col = 3, 4
cell = raw_output[row, col]  # shape: (output_depth,)

# Box 1: indices 0-4
box1_cx   = torch.sigmoid(cell[0]).item()
box1_cy   = torch.sigmoid(cell[1]).item()
box1_w    = torch.sigmoid(cell[2]).item()
box1_h    = torch.sigmoid(cell[3]).item()
box1_conf = torch.sigmoid(cell[4]).item()

# Box 2: indices 5-9
box2_cx   = torch.sigmoid(cell[5]).item()
box2_cy   = torch.sigmoid(cell[6]).item()
box2_w    = torch.sigmoid(cell[7]).item()
box2_h    = torch.sigmoid(cell[8]).item()
box2_conf = torch.sigmoid(cell[9]).item()

# Class scores: indices 10 onwards
class_probs = torch.softmax(cell[10:], dim=0)
pred_class  = class_probs.argmax().item()

print(f'Cell ({row}, {col}) predictions:')
print(f'  Box 1: cx={box1_cx:.3f}  cy={box1_cy:.3f}  w={box1_w:.3f}  h={box1_h:.3f}  conf={box1_conf:.3f}')
print(f'  Box 2: cx={box2_cx:.3f}  cy={box2_cy:.3f}  w={box2_w:.3f}  h={box2_h:.3f}  conf={box2_conf:.3f}')
print(f'  Predicted class: {classes[pred_class]}  (score={class_probs[pred_class]:.3f})')
print(f'  Final confidence box1: {box1_conf * class_probs[pred_class].item():.3f}')

## Part 4 — Non-Maximum Suppression

NMS removes duplicate detections. Given a set of predicted boxes with confidence scores,
it keeps the highest-confidence box and removes any box with IoU > threshold against it.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    boxes:  list of (x_min, y_min, x_max, y_max)
    scores: list of confidence scores
    returns: indices of kept boxes
    """
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    kept = []
    while order:
        best = order.pop(0)
        kept.append(best)
        order = [
            i for i in order
            if iou(boxes[best], boxes[i])[0] < iou_threshold
        ]
    return kept

# Five overlapping predictions for the same object
boxes = [
    (1.0, 1.0, 5.0, 4.0),
    (1.2, 1.1, 5.2, 4.1),
    (0.9, 0.8, 4.9, 3.8),
    (1.5, 1.5, 5.5, 4.5),
    (6.0, 5.0, 9.0, 8.0),  # different object — should be kept
]
scores = [0.91, 0.85, 0.78, 0.62, 0.88]

kept = nms(boxes, scores, iou_threshold=0.5)

print('All boxes and scores:')
for i, (b, s) in enumerate(zip(boxes, scores)):
    print(f'  Box {i}: {b}  score={s}')

print(f'\nBoxes kept after NMS (threshold=0.5): {kept}')
for i in kept:
    print(f'  Box {i}: {boxes[i]}  score={scores[i]}')

## Summary

This notebook demonstrated:

1. **Classification pipeline** — a two-block CNN on CIFAR-10 showing the full path from raw image to softmax probabilities to argmax prediction; the probability bar charts make the confidence visible
2. **IoU calculation** — intersection over union computed step by step, matching the Exercise 23 Section C worked example (IoU = 12/36 = 0.333, below the 0.5 threshold)
3. **YOLO output tensor** — the S×S×(B×5+C) structure parsed for a single grid cell, showing how cx, cy, w, h, confidence, and class scores are read from the flat vector
4. **Non-maximum suppression** — duplicate boxes for the same object removed by keeping the highest-confidence box and discarding overlapping boxes above the IoU threshold